# Compute index-level confidence intervals

Compute spatial-autocorrelation-adjusted intervals for each site, index, month, and observation year. Each row represents one raster observation; no year-pair differences are formed. For each site-month-index group, every year uses the intersection of valid pixels across the configured years.

In [1]:
from pathlib import Path

import pandas as pd
import rasterio

from index_ci import IndexAnalysisConfig, IndexCIAnalysis, discover_observations

ROOT = Path.cwd().parent
assert (ROOT / 'pyproject.toml').exists(), 'Run notebooks from the notebooks/ folder'
config = IndexAnalysisConfig.from_yaml(
    ROOT / 'configs' / 'index_ci.yaml', project_root=ROOT
)
config.to_dict()

{'data_root': '/home/tshihadah/code/wadi-gaza-research/outputs/composites',
 'output_dir': '/home/tshihadah/code/wadi-gaza-research/outputs/index_ci',
 'study_areas': None,
 'months': ('01', '05', '09', '11'),
 'year_range': (2019, 2025),
 'index_bands': ('NDVI', 'GNDVI', 'SAVI', 'NDRE'),
 'engine': 'fft',
 'tau': 0.05,
 'tau_sensitivity': (0.02, 0.05, 0.1),
 'alpha': 0.05,
 'max_lag_fraction': 0.25,
 'min_n_eff': 30.0}

## Discover and validate observations

The expected count is one raster for each configured site × month × year. This check prevents an incomplete time series from silently changing the shared spatial support.

In [2]:
observations = discover_observations(config)
if not observations:
    raise ValueError('No raster observations matched the configured filters')
study_areas = config.study_areas or tuple(sorted({item.study_area for item in observations}))
expected_keys = {
    (area, month, year)
    for area in study_areas
    for month in config.months
    for year in range(config.year_range[0], config.year_range[1] + 1)
}
observation_keys = [(item.study_area, item.month, item.year) for item in observations]
missing_keys = expected_keys - set(observation_keys)
duplicate_keys = len(observation_keys) - len(set(observation_keys))
if missing_keys or duplicate_keys:
    raise ValueError(
        f'Missing observations: {sorted(missing_keys)[:10]}; duplicate keys: {duplicate_keys}'
    )
missing_bands = []
for observation in observations:
    with rasterio.open(observation.path) as raster:
        missing = set(config.index_bands) - set(raster.descriptions)
    if missing:
        missing_bands.append((observation.path, sorted(missing)))
if missing_bands:
    raise ValueError(f'Configured index bands missing from composites: {missing_bands[:10]}')
observation_table = pd.DataFrame([
    {
        'study_area': item.study_area,
        'month': item.month,
        'year': item.year,
        'path': item.path,
    }
    for item in observations
])
display(observation_table)

,study_area,month,year,path
0,AlAhrashProtectorate,01,2019,/home/tshihadah/code/wadi-gaza-research/output...
1,AlAhrashProtectorate,05,2019,/home/tshihadah/code/wadi-gaza-research/output...
2,AlAhrashProtectorate,09,2019,/home/tshihadah/code/wadi-gaza-research/output...
3,AlAhrashProtectorate,11,2019,/home/tshihadah/code/wadi-gaza-research/output...
4,AlAhrashProtectorate,01,2020,/home/tshihadah/code/wadi-gaza-research/output...
...,...,...,...,...
191,WadiGaza,11,2024,/home/tshihadah/code/wadi-gaza-research/output...
192,WadiGaza,01,2025,/home/tshihadah/code/wadi-gaza-research/output...
193,WadiGaza,05,2025,/home/tshihadah/code/wadi-gaza-research/output...
194,WadiGaza,09,2025,/home/tshihadah/code/wadi-gaza-research/output...


## Run and save

The module writes separate results, correlograms, and config snapshots under `outputs/index_ci/<INDEX>/`.

In [3]:
analysis = IndexCIAnalysis(config)
results, correlograms = analysis.run(observations)
analysis.save(results, correlograms)
print(
    f'Saved {len(results)} result rows and {len(correlograms)} correlogram rows '
    f'under {config.output_dir}'
)
summary = results.groupby('index').agg(
    result_rows=('index', 'size'),
    failed_observations=('error', lambda values: values.fillna('').ne('').sum()),
)
display(summary)
expected_rows = len(observations) * len(config.tau_sensitivity)
if not summary['result_rows'].eq(expected_rows).all() or summary['failed_observations'].sum():
    raise RuntimeError('At least one index has incomplete or failed observations; inspect saved results.')

Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


... (776 outputs left out)

Raster pixels are not square: (10.000130544932942, 10.00695479149712)


Raster pixels are not square: (10.000130544932942, 10.00695479149712)


Saved 2352 result rows and 58688 correlogram rows under /home/tshihadah/code/wadi-gaza-research/outputs/index_ci


,result_rows,failed_observations
index,,
GNDVI,588,0
NDRE,588,0
NDVI,588,0
SAVI,588,0


## Primary-threshold results and review flags

In [4]:
primary = results.loc[results['is_primary_tau']].copy()
summary_columns = [
    'study_area', 'index', 'month', 'year', 'n', 'mean_index', 'sd',
    'range_px', 'range_map_units', 'n_eff', 'ci_low', 'ci_high',
    'naive_ci_low', 'naive_ci_high', 'flag_no_range', 'flag_low_n_eff', 'error',
]
display(primary[summary_columns].sort_values(['index', 'study_area', 'month', 'year']))
review = primary.loc[
    primary['flag_no_range']
    | primary['flag_low_n_eff']
    | primary['error'].fillna('').ne('')
]
print(f'{len(review)} primary-threshold observation(s) flagged or failed.')
display(review[summary_columns].sort_values(['index', 'study_area', 'month', 'year']))

,study_area,index,month,year,n,mean_index,sd,range_px,range_map_units,n_eff,ci_low,ci_high,naive_ci_low,naive_ci_high,flag_no_range,flag_low_n_eff,error
589,AlAhrashProtectorate,GNDVI,01,2019,33848,0.310632,0.113981,48.0,479.969706,26.240627,0.264828,0.356436,0.309418,0.311846,False,True,
592,AlAhrashProtectorate,GNDVI,01,2020,33848,0.335112,0.117708,46.0,459.970968,27.210994,0.288747,0.381477,0.333858,0.336366,False,True,
595,AlAhrashProtectorate,GNDVI,01,2021,33848,0.311715,0.111623,42.0,419.973493,31.721251,0.271279,0.352151,0.310526,0.312904,False,False,
598,AlAhrashProtectorate,GNDVI,01,2022,33848,0.342339,0.129358,44.0,439.972231,28.751850,0.292901,0.391776,0.340960,0.343717,False,True,
601,AlAhrashProtectorate,GNDVI,01,2023,33848,0.330717,0.108629,45.0,449.971599,27.195553,0.287915,0.373519,0.329559,0.331874,False,True,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1750,WadiGaza,SAVI,11,2021,28413,0.196656,0.105898,32.0,320.004177,87.887316,0.174204,0.219108,0.195425,0.197887,False,False,
1753,WadiGaza,SAVI,11,2022,28413,0.193821,0.105582,33.0,330.004308,86.864217,0.171300,0.216341,0.192593,0.195048,False,False,
1756,WadiGaza,SAVI,11,2023,28413,0.194161,0.098139,35.0,350.004569,72.294802,0.171149,0.217174,0.193020,0.195303,False,False,
1759,WadiGaza,SAVI,11,2024,28413,0.117402,0.048709,15.0,150.001958,244.473226,0.111266,0.123538,0.116836,0.117968,False,False,


251 primary-threshold observation(s) flagged or failed.


,study_area,index,month,year,n,mean_index,sd,range_px,range_map_units,n_eff,ci_low,ci_high,naive_ci_low,naive_ci_high,flag_no_range,flag_low_n_eff,error
589,AlAhrashProtectorate,GNDVI,01,2019,33848,0.310632,0.113981,48.0,479.969706,26.240627,0.264828,0.356436,0.309418,0.311846,False,True,
592,AlAhrashProtectorate,GNDVI,01,2020,33848,0.335112,0.117708,46.0,459.970968,27.210994,0.288747,0.381477,0.333858,0.336366,False,True,
598,AlAhrashProtectorate,GNDVI,01,2022,33848,0.342339,0.129358,44.0,439.972231,28.751850,0.292901,0.391776,0.340960,0.343717,False,True,
601,AlAhrashProtectorate,GNDVI,01,2023,33848,0.330717,0.108629,45.0,449.971599,27.195553,0.287915,0.373519,0.329559,0.331874,False,True,
604,AlAhrashProtectorate,GNDVI,01,2024,33848,0.327772,0.102134,42.0,419.973493,29.274192,0.289122,0.366423,0.326684,0.328861,False,True,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1606,NahalRubin,SAVI,01,2022,7860,0.278115,0.167921,NaN,NaN,NaN,NaN,NaN,0.274402,0.281828,True,False,
1609,NahalRubin,SAVI,01,2023,7860,0.276564,0.137109,NaN,NaN,NaN,NaN,NaN,0.273533,0.279596,True,False,
1612,NahalRubin,SAVI,01,2024,7860,0.271188,0.146734,NaN,NaN,NaN,NaN,NaN,0.267944,0.274432,True,False,
1615,NahalRubin,SAVI,01,2025,7860,0.260575,0.136715,NaN,NaN,NaN,NaN,NaN,0.257552,0.263598,True,False,
